In [1]:
import os
import mne
from mne.preprocessing import ICA
import mne_bids
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

In [ ]:
# --- CONFIGURATION ---
bids_root = r"ds004395" # Update this path to where your dataset is located in Colab
'''subject_id = 'LTP064'''
task = 'ltpFR'
'''session_id = '8'''

# Define a new output directory for your clean data
output_dir = r"preprocessed_ica"
file_name = r"scaled_data_multiclass_task.npz"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory set to: {output_dir}")

all_subjects = ['LTP064', 'LTP066', 'LTP067', 'LTP069']
''' --------  ['LTP063', 'LTP064', 'LTP065', 'LTP066', 'LTP067', 'LTP068', 'LTP069', 'LTP070']  ----------'''
all_sessions = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']
''' --------   # Add all session IDs here  ----------'''

all_X_data = []
all_Y_data = []


Output directory set to: C:/Users/Jaya Prakash/Desktop/EEGData/preprocessed_ica/


In [11]:
# --- HELPER FUNCTIONS ---

def preprocess_raw(raw):
    """
    Applies optimized preprocessing with ICA for artifact correction:
    1. Downsample the data to 250 Hz
    2. Sets EOG types
    3. High-pass filters (0.5 Hz)
    4. Fits ICA on a 1.0 Hz filtered copy
    5. Finds and removes EOG (blink) components
    6. Removes the 129th channel
    7. Applies Common Average Reference (CAR)
    (No low-pass filter is applied)
    """

    # CRITICAL: Ensure data is loaded into memory
    raw.load_data()

    # 1. Downsample to 250 Hz
    # WARNING: Downsampling without a prior low-pass filter
    # (below 125 Hz) will cause aliasing artifacts.
    print(f"    ... Downsampling from {raw.info['sfreq']} Hz to 250 Hz")
    raw.resample(sfreq=250, verbose=False)

    # 2. Handle EOG Channels (Based on previous warnings)
    eog_channels = ['E8', 'E26', 'E126', 'E127']
    raw.set_channel_types({ch: 'eog' for ch in eog_channels})

    # 3. High-pass filter the main data at 0.5 Hz
    raw.filter(l_freq=0.5, h_freq=None, fir_design='firwin', verbose=False)

    # 4. Fit ICA on a 1.0 Hz filtered copy
    print("    ... Fitting ICA on 1.0 Hz filtered data")
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, fir_design='firwin', verbose=False)

    ica = ICA(n_components=30, random_state=42, max_iter='auto')
    ica.fit(raw_for_ica, verbose=False)

    # 5. Find and remove EOG (blink) components
    print("    ... Finding EOG components")
    eog_indices, eog_scores = ica.find_bads_eog(raw_for_ica, ch_name=eog_channels)

    if eog_indices:
        print(f"    ... Found {len(eog_indices)} EOG components: {eog_indices}")
        ica.exclude = eog_indices

        # Apply the ICA solution to the *original* 0.5 Hz filtered data
        print("    ... Removing EOG components")
        ica.apply(raw, verbose=False)
    else:
        print("    ... No EOG components found.")

    # 6. Low-pass filter (REMOVED)
    # raw.filter(l_freq=None, h_freq=40.0, fir_design='firwin', verbose=False)

    # 7. Remove the 129th channel
    ref_channel_name = raw.ch_names[-1]
    print(f"    ... Dropping channel: {ref_channel_name}")
    raw.drop_channels([ref_channel_name])

    # 8. Apply Common Average Reference (CAR)
    print("    ... Applying Common Average Reference (CAR)")
    raw.set_eeg_reference('average', projection=False, verbose=False)

    return raw


In [ ]:
def label_and_epoch_data_multiclass(raw, subject_id, session_id):
    """
    Epochs EEG data around events and assigns binary class labels
    based on the "task" column carried in raw.annotations
    (read by mne_bids from the BIDS events.tsv "task" column).

    "task" = type of judgment associated with the event:
         0 -> Size     -> label 0
         1 -> Animacy  -> label 1

    All annotations (regardless of "description") whose "task" is
    0 or 1 are kept; rows where "task" is NaN/n-a or -1 are excluded.

    If multiple annotations map to the same EEG sample index after
    rounding (duplicate event times), only the first one (by original
    order) is kept, so events and Y_df stay perfectly aligned with X.
    """
    print(f"Entered label_and_epoch_data_multiclass function")

    TASK_MAP = {0: 0, 1: 1}
    label_to_name = {0: "Size", 1: "Animacy"}

    # -- Pull annotations (mne_bids attaches events.tsv columns,
    #    including "task", as extra columns here) --
    full_metadata = raw.annotations.to_data_frame()
    full_metadata["item_name"] = full_metadata["item_name"].astype(str)
    full_metadata["subject"]   = subject_id
    full_metadata["session"]   = session_id
    full_metadata["onset_sec"] = raw.annotations.onset   # float, seconds
    
    # -- Filter to annotations that carry task == 0 or 1 (drop NaN and -1/Control) --
    task_numeric = pd.to_numeric(full_metadata["task"], errors="coerce")
    valid_mask = task_numeric.isin(list(TASK_MAP.keys()))

    task_df = full_metadata[valid_mask].reset_index(drop=True).copy()
    task_df["task"] = task_numeric[valid_mask].astype(int).values

    if task_df.empty:
        raise RuntimeError(
            "No annotations with a valid 'task' value (0 or 1) found. "
            f"Available descriptions: {full_metadata['description'].unique().tolist()}"
        )

    task_df["class_label"] = task_df["task"].map(TASK_MAP)
    task_df["event_type"]  = task_df["class_label"].map(label_to_name)

    print(f"\nEvent counts for this session (by task-derived class):")
    print(task_df["event_type"].value_counts())
    # ------------  print(f"\nBreakdown by original annotation description:")
    # ------------  print(task_df.groupby(["description", "event_type"]).size())

    # ------------  task_df.to_csv("task_df.csv", index=False)
    # ------------  print(f"the final task dataframe is downloaded and saved to task_df.csv")

    # -- Compute sample index for each annotation --
    sfreq      = raw.info["sfreq"]
    first_samp = raw.first_samp

    task_df["sample"] = np.round(task_df["onset_sec"].values * sfreq).astype(int) + first_samp

    # -- Deduplicate: if multiple annotations land on the same sample,
    #    keep only the first (by original order) so events/Y_df stay
    #    aligned with the epochs MNE actually creates --
    n_before = len(task_df)
    task_df = task_df.drop_duplicates(subset="sample", keep="first").reset_index(drop=True)
    n_after = len(task_df)
    if n_after < n_before:
        print(f"\n⚠ Dropped {n_before - n_after} duplicate-sample annotations "
              f"(same EEG timepoint, multiple events).")

    # -- Build MNE events array directly from sample positions --
    events = np.column_stack([
        task_df["sample"].values,
        np.zeros(len(task_df), dtype=int),
        task_df["class_label"].values.astype(int),
    ])

    # mne.Epochs requires events sorted by sample
    sort_idx = np.argsort(events[:, 0])
    events = events[sort_idx]
    Y_df = task_df.iloc[sort_idx].reset_index(drop=True)

    present_codes = np.unique(events[:, 2])
    event_id = {label_to_name[c]: int(c) for c in present_codes}
    # ----------  print(f"\nMNE event_id dict: {event_id}")
    print(f"MNE events array shape: {events.shape}")

    # -- Epoch --
    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=-0.2,
        tmax=1.0,
        preload=True,
        baseline=(-0.2, 0),
        verbose=False,
    )

    X = epochs.get_data()
    print(f"\nX shape: {X.shape}")

    # -- Sanity checks --
    if len(X) != len(Y_df):
        raise RuntimeError(
            f"X/Y mismatch: X={len(X)} epochs, Y_df={len(Y_df)} rows. "
            f"MNE likely dropped edge epochs. Check tmin/tmax window."
        )

    mne_codes = epochs.events[:, 2]
    y_codes   = Y_df["class_label"].values
    if not np.array_equal(mne_codes, y_codes):
        raise RuntimeError(
            "Class label order does not match MNE epoch order. "
            "Check onset matching logic."
        )

    Y_df = Y_df[["onset_sec", "item_name", "subject", "session",
                  "task", "event_type", "class_label", "description"]].rename(
        columns={"onset_sec": "onset"}
    )

    print(f"\nSuccess - returning X: {X.shape}, Y_df: {Y_df.shape}")
    return X, Y_df

In [ ]:

# --- MAIN LOOP (Nested) ---

# Outer loop for subjects
for subject_id in all_subjects:
    print(f"Processing Subject: {subject_id}")

    # Inner loop for sessions
    for session_id in all_sessions:
        print(f"\n--- Processing Session {session_id} ---")

        try:
            bids_path = mne_bids.BIDSPath(
                subject=subject_id, session=session_id, datatype='eeg',
                task=task, root=bids_root
            )
            print(f" the file path is :: {bids_path}")
            raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False, extra_params={'preload': True})
            print(f"    Loaded raw data: {raw.info['nchan']} channels, {raw.info['sfreq']} Hz.")

            raw_preprocessed = preprocess_raw(raw)
            print("The EEG data is preprocessed successfully")

            X_session, Y_session = label_and_epoch_data_multiclass(
                raw_preprocessed, subject_id, session_id
            )

            # Add this session's data to the subject's lists
            all_X_data.append(X_session)
            all_Y_data.append(Y_session)

            print(f"    Session {session_id} successful. Extracted {len(X_session)} epochs.")

        except FileNotFoundError:
            print(f"\u274c File not found for Subject {subject_id}, Session {session_id}. Skipping.")
        except Exception as e:
            print(f"\u274c An error occurred for Subject {subject_id}, Session {session_id}: {e}")

Processing Subject: LTP064

--- Processing Session 6 ---
 the file path is :: C:/Users/Jaya Prakash/Desktop/EEGData/ds004395/sub-LTP064/ses-6/eeg/sub-LTP064_ses-6_task-ltpFR_eeg.edf


C:\Users\Jaya Prakash\AppData\Local\Temp\ipykernel_21020\1121968452.py:17: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['E8', 'E25', 'E126', 'E127']
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False, extra_params={'preload': True},on_ch_mismatch='rename') # -- Need to remove this parameter, while uploading to singularity.
C:\Users\Jaya Prakash\AppData\Local\Temp\ipykernel_21020\1121968452.py:17: RuntimeWarning: Not setting positions of 4 eog channels found in montage:
['E8', 'E25', 'E126', 'E127']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False, extra_params={'preload': True},on_ch_mismatch='rename') # -- Need to remove this parameter, while uploading to singularity.
C:\Users\Jaya Prakash\AppData\Local\Temp\ipykernel_21020\112196845

    Loaded raw data: 129 channels, 500.0 Hz.
    Channels found in raw data after loading: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E32', 'E33', 'E34', 'E35', 'E36', 'E37', 'E38', 'E39', 'E40', 'E41', 'E42', 'E43', 'E44', 'E45', 'E46', 'E47', 'E48', 'E49', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E63', 'E64', 'E65', 'E66', 'E67', 'E68', 'E69', 'E70', 'E71', 'E72', 'E73', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E81', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E88', 'E89', 'E90', 'E91', 'E92', 'E93', 'E94', 'E95', 'E96', 'E97', 'E98', 'E99', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E107', 'E108', 'E109', 'E110', 'E111', 'E112', 'E113', 'E114', 'E115', 'E116', 'E117', 'E118', 'E119', 'E120', 'E121', 'E122', 'E123', 'E124', 'E125', 'E126', 'E127', 

In [ ]:
# --- PROCESS AND SAVE (MULTI-CLASS: task column) ---
if all_X_data:

    # 1. Combine all sessions for this subject
    X_combined = np.concatenate(all_X_data, axis=0)
    Y_combined = pd.concat(all_Y_data, ignore_index=True)

    print(f"Total epochs across all sessions: {len(X_combined)}")
    print(f"\nClass distribution (raw counts):")
    print(Y_combined["event_type"].value_counts())

    # 2. Define class map and extract integer labels
    #    This must match the TASK_MAP used in label_and_epoch_data_multiclass()
    #    Original "task" column values: 0 (Size), 1 (Animacy)
    CLASS_MAP = {
        "Size":    0,
        "Animacy": 1
    }
    CLASS_NAMES = np.array(["Size", "Animacy"])

    target_label = "class_label"
    y = Y_combined[target_label].values    # integer array, values in {0, 1, 2}

    # Verify all expected classes are present
    unique_classes = np.unique(y)
    print(f"\nClasses present in this subject's data: "
          f"{[CLASS_NAMES[i] for i in unique_classes]}")

    # 3. Train/test split
    #    stratify=y still works for 2 classes — sklearn handles N-class stratification.
    #    If any class has fewer than 2 samples the split will fail; the check below
    #    catches that before it crashes.
    class_counts = np.bincount(y, minlength=len(CLASS_NAMES))
    too_few = [CLASS_NAMES[i] for i, c in enumerate(class_counts) if 0 < c < 2]
    if too_few:
        print(f"  \u26a0 Warning: classes {too_few} have <2 samples. "
              f"  Stratified split may fail \u2014 consider merging or dropping them.")

    X_train, X_test, y_train, y_test = train_test_split(
        X_combined, y,
        test_size=0.30,
        stratify=y,           # preserves 2-class proportions in both splits
        random_state=42,
    )

    print(f"\nSplit sizes \u2014 train: {len(X_train)}  test: {len(X_test)}")
    print("Train class counts:", np.bincount(y_train, minlength=len(CLASS_NAMES)))
    print("Test  class counts:", np.bincount(y_test,  minlength=len(CLASS_NAMES)))

    # 4. Apply RobustScaler (fit on train only — no leakage)
    scaler = RobustScaler()
    n_trials_train, n_chans, n_times = X_train.shape

    # Reshape to 2D for scaler: (n_trials \u00d7 n_chans, n_times)
    X_train_2d = X_train.reshape(n_trials_train * n_chans, n_times)

    print("\n    Fitting RobustScaler on training data only...")
    scaler.fit(X_train_2d)

    # Transform train
    X_train_scaled = scaler.transform(X_train_2d).reshape(n_trials_train, n_chans, n_times)

    # Transform test (using the same scaler fitted on train)
    n_trials_test  = X_test.shape[0]
    X_test_scaled  = scaler.transform(
        X_test.reshape(n_trials_test * n_chans, n_times)
    ).reshape(n_trials_test, n_chans, n_times)

    print("    Data standardized.")

    # 5. Save to .npz
    #    class_names is saved alongside the arrays so the file is self-contained.
    #    When you load this file you never need to remember the class order:
    #        data = np.load(file, allow_pickle=True)
    #        class_names = data["class_names"]   \u2192  ["Size", "Animacy"]
    output_filename = os.path.join(output_dir, file_name)
    print(f"\n    Saving to {output_filename} ...")

    np.savez_compressed(
        output_filename,
        X_train=X_train_scaled.astype(np.float32),   # float32 saves ~50% disk vs float64
        X_test=X_test_scaled.astype(np.float32),
        y_train=y_train,
        y_test=y_test,
        class_names=CLASS_NAMES,                      # saved as part of the file
        class_map=np.array(list(CLASS_MAP.items())),  # human-readable key\u2192int mapping
    )

    print(f"\u2705 Saved multi-class (task) data.")
    print(f"   Arrays inside the .npz:")
    print(f"     X_train : {X_train_scaled.shape}  (n_trials, n_channels, n_times)")
    print(f"     X_test  : {X_test_scaled.shape}")
    print(f"     y_train : {y_train.shape}  values in {np.unique(y_train)}")
    print(f"     y_test  : {y_test.shape}")
    print(f"     class_names : {CLASS_NAMES}")

else:
    print(f"--- No data processed. No files saved. ---")

print("\nAll subjects preprocessed and saved!")

Total epochs across all sessions: 741

Class distribution (raw counts):
event_type
Size       372
Animacy    369
Name: count, dtype: int64

Classes present in this subject's data: [np.str_('Size'), np.str_('Animacy')]

Split sizes — train: 518  test: 223
Train class counts: [260 258]
Test  class counts: [112 111]

    Fitting RobustScaler on training data only...
    Data standardized.

    Saving to C:/Users/Jaya Prakash/Desktop/EEGData/preprocessed_ica/scaled_data_multiclass_task.npz ...
✅ Saved multi-class (task) data.
   Arrays inside the .npz:
     X_train : (518, 128, 551)  (n_trials, n_channels, n_times)
     X_test  : (223, 128, 551)
     y_train : (518,)  values in [0 1]
     y_test  : (223,)
     class_names : ['Size' 'Animacy']

All subjects preprocessed and saved!
